In [1]:
# Import av nødvendige biblioteker
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import pytz 
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

# Konstanter for Spark/Cassandra
CASSANDRA_HOST = "127.0.0.1"  # Erstatt om nødvendig
CASSANDRA_KEYSPACE = "elhub_data"

try:
    # Sørg for at riktig connector-versjon er spesifisert og tilgjengelig
    spark = (
        SparkSession.builder.appName("ElhubDataPipeline")
        .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.4.1") 
        .config("spark.cassandra.connection.host", CASSANDRA_HOST)
        .getOrCreate()
    )
    print("✅ SparkSession opprettet og konfigurert med Cassandra-tilkobling.")
except Exception as e:
    print(f"❌ Feil ved oppretting av SparkSession: {e}")
    spark = None

if spark:
    try:
        from cassandra.cluster import Cluster
        cluster = Cluster([CASSANDRA_HOST])
        session = cluster.connect()
        
        session.execute(f"""
            CREATE KEYSPACE IF NOT EXISTS {CASSANDRA_KEYSPACE}
            WITH REPLICATION = {{ 'class': 'SimpleStrategy', 'replication_factor': 1 }};
        """)
        session.set_keyspace(CASSANDRA_KEYSPACE)
        
        # Tabell-skjema som matcher dataen fra API
        session.execute(f"""
            CREATE TABLE IF NOT EXISTS {CASSANDRA_KEYSPACE}.elhub_production (
                priceArea text,
                productionGroup text,
                startTime text,
                endTime text,
                quantityKwh double,
                measurementTime text,
                PRIMARY KEY (priceArea, productionGroup, startTime)
            );
        """)
        print(f"✅ Cassandra KEYSPACE '{CASSANDRA_KEYSPACE}' og tabell 'elhub_production' satt opp.")
        
        session.shutdown()
        cluster.shutdown()
        
    except Exception as e:
        print(f"❌ Kritisk feil under Cassandra-tilkobling/Keyspace-oppretting: {e}")
        print("Sjekk at Cassandra kjører på CASSANDRA_HOST.")

:: loading settings :: url = jar:file:/Users/a.h.sheikh/.pyenv/versions/3.12.6/envs/ind320/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/a.h.sheikh/.ivy2/cache
The jars for the packages stored in: /Users/a.h.sheikh/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c1fcc93f-9d93-42df-8b14-3f4bc50aba24;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.4.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.4.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-strea

✅ SparkSession opprettet og konfigurert med Cassandra-tilkobling.
✅ Cassandra KEYSPACE 'elhub_data' og tabell 'elhub_production' satt opp.


In [2]:
# Import av nødvendige biblioteker (fortsettelse fra Steg 1)
import pandas as pd
import requests
from datetime import datetime, timedelta
import pytz 
import json
import time

# --- API Konfigurasjon ---
ELHUB_API_URL = "https://api.elhub.no/energy-data/v0/price-areas" 
DATA_TYPE = "PRODUCTION_PER_GROUP_MBA_HOUR"
YEAR = 2023 # Bruker 2023 da dette ga data i debug
TIMEZONE = pytz.timezone('Europe/Oslo') 
START_DATE_YEAR = datetime(YEAR, 1, 1, 0, 0, tzinfo=TIMEZONE)
END_DATE_YEAR = datetime(YEAR, 12, 31, 23, 59, tzinfo=TIMEZONE)

all_production_data = []
current_date = START_DATE_YEAR
request_count = 0
MAX_DAYS_PER_REQUEST = 7 

print(f"--- Starter ukentlig innhenting av {DATA_TYPE} for året {YEAR} (KORRIGERT) ---")

while current_date <= END_DATE_YEAR:
    
    start_time_local = current_date
    end_time_local = current_date + timedelta(days=MAX_DAYS_PER_REQUEST) - timedelta(minutes=1)
    if end_time_local > END_DATE_YEAR:
        end_time_local = END_DATE_YEAR
        
    start_time = start_time_local.isoformat()
    end_time = end_time_local.isoformat()
    
    request_count += 1
    
    # 1. Ett kall henter alle prisområder for perioden
    params = {
        "dataset": DATA_TYPE,
        # FJERNER 'id': area fra params. Kaller uten ID for å få alle
        "startDate": start_time,
        "endDate": end_time
    }

    print(f"\n> Henter data for periode: {start_time_local.strftime('%Y-%m-%d')} til {end_time_local.strftime('%Y-%m-%d')} (ALLE områder)")

    try:
        response = requests.get(ELHUB_API_URL, params=params, timeout=20) 
        response.raise_for_status() 

        data = response.json()
        posts_in_period = 0
        
        # 2. ITERERER over data-arrayet (JSON:API-format)
        for item in data.get("data", []):
            attributes = item.get("attributes", {})
            area_name = attributes.get("name") # Får NO1, NO2 osv.
            
            # Den faktiske tidsseriedataen ligger her:
            production_list_for_area = attributes.get("productionPerGroupMbaHour", [])
            
            if production_list_for_area:
                posts_in_period += len(production_list_for_area)
                all_production_data.extend(production_list_for_area)
            
            # Vi kan bruke denne utskriften for å bekrefte at alle områdene prosesseres
            print(f"  > {area_name}: Hentet {len(production_list_for_area)} poster.")


        if posts_in_period > 0:
            print(f"  ✅ Vellykket: Totalt {posts_in_period} poster hentet i perioden.")
        else:
            print("  ⚠️ Ingen produksjonsposter funnet i denne perioden.")

    except requests.exceptions.HTTPError as e:
        print(f"  ❌ HTTP Feil ({response.status_code}): API returnerte en feil.")
    except requests.exceptions.RequestException as e:
        print(f"  ❌ Tilkoblingsfeil: {e}")
            
    # Gå til starten av neste 7-dagers intervall
    current_date += timedelta(days=MAX_DAYS_PER_REQUEST)
    time.sleep(0.5) # Forsikre at vi ikke treffer rate-limit

print("\n--- Datainnhenting fullført ---")
print(f"Totalt antall rådata-poster hentet: {len(all_production_data)}")

# Konverter listen til Pandas DataFrame
if all_production_data:
    df_raw = pd.DataFrame(all_production_data)
    # Kolonnen 'priceArea' mangler i de individuelle listene, så vi må legge den til.
    # Dette er IKKE ideelt, men kan fungere hvis den er implisitt i responsen.
    # JSON:API-strukturen er komplisert. Vi tar en sjanse: hvis API-et ikke legger til
    # 'priceArea' i de indre objektene, må vi re-introduere loopen.
    # For nå, bruk det vi har:
    REQUIRED_COLUMNS = ['productionGroup', 'startTime', 'endTime', 'quantityKwh', 'measurementTime'] # 'priceArea' mangler her!
    
    # KUN FORSIKRER AT VI KAN FORTSETTE MED RESTEN AV FLYTEN
    df_cassandra = pd.DataFrame(all_production_data)
    # Vi MÅ legge til priceArea fra 'name'-feltet!
    print("⚠️ VIKTIG: Den nye API-strukturen krever at vi trekker ut 'name' (priceArea) fra metadata og legger til i hver rad.")
    print("Vi fortsetter ved å anta at dataene KUN har de indre kolonnene for nå.")
    
    # Vi må dessverre tilbake til å loope for å koble dataen til rett prisområde
    print("❌ Feilaktig antakelse. Går tilbake til looping for å koble data til prisområde...")
    df_cassandra = None # Tvinger oss til å bruke neste kodebit
else:
    print("❌ Ingen data å konvertere til DataFrame. Kan ikke fortsette.")
    df_cassandra = pd.DataFrame()

--- Starter ukentlig innhenting av PRODUCTION_PER_GROUP_MBA_HOUR for året 2023 (KORRIGERT) ---

> Henter data for periode: 2023-01-01 til 2023-01-07 (ALLE områder)
  > *: Hentet 0 poster.
  > NO1: Hentet 840 poster.
  > NO2: Hentet 840 poster.
  > NO3: Hentet 840 poster.
  > NO4: Hentet 840 poster.
  > NO5: Hentet 840 poster.
  ✅ Vellykket: Totalt 4200 poster hentet i perioden.

> Henter data for periode: 2023-01-08 til 2023-01-14 (ALLE områder)
  > *: Hentet 0 poster.
  > NO1: Hentet 840 poster.
  > NO2: Hentet 840 poster.
  > NO3: Hentet 840 poster.
  > NO4: Hentet 840 poster.
  > NO5: Hentet 840 poster.
  ✅ Vellykket: Totalt 4200 poster hentet i perioden.

> Henter data for periode: 2023-01-15 til 2023-01-21 (ALLE områder)
  > *: Hentet 0 poster.
  > NO1: Hentet 840 poster.
  > NO2: Hentet 840 poster.
  > NO3: Hentet 840 poster.
  > NO4: Hentet 840 poster.
  > NO5: Hentet 840 poster.
  ✅ Vellykket: Totalt 4200 poster hentet i perioden.

> Henter data for periode: 2023-01-22 til 2023

In [3]:
# --- Kjører Steg 3 (Last data til Cassandra) ---

# Re-opprett SparkSession i tilfelle den stoppet (på grunn av advarslene)
try:
    spark = (
        SparkSession.builder.appName("ElhubDataPipeline")
        .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.4.1") 
        .config("spark.cassandra.connection.host", CASSANDRA_HOST)
        .getOrCreate()
    )
    print("✅ SparkSession re-opprettet og konfigurert.")
except Exception as e:
    print(f"❌ Feil ved re-oppretting av SparkSession: {e}")
    spark = None
    
# Gjenbruker df_cassandra fra Steg 2 (som nå er fylt med data)

if spark and not df_cassandra.empty:
    try:
        # Konverter Pandas DataFrame til Spark DataFrame
        spark_df_raw = spark.createDataFrame(df_cassandra)
        
        print("\n--- Starter lasting av data til Cassandra ---")
        
        # Sletter eksisterende data før lasting for å unngå duplikater
        spark.read.format("org.apache.spark.sql.cassandra").options(table="elhub_production", keyspace=CASSANDRA_KEYSPACE).load().write \
            .format("org.apache.spark.sql.cassandra") \
            .options(table="elhub_production", keyspace=CASSANDRA_KEYSPACE) \
            .mode("overwrite") \
            .save()
            
        print("✅ Data lastet inn i Cassandra-tabellen 'elhub_production'.")
        
        # Verifisering: Les data tilbake fra Cassandra for videre bruk
        df_cassandra_read = spark.read \
            .format("org.apache.spark.sql.cassandra") \
            .options(table="elhub_production", keyspace=CASSANDRA_KEYSPACE) \
            .load()
            
        print(f"✅ Verifisering: Antall rader lest tilbake fra Cassandra: {df_cassandra_read.count()}")
        
    except Exception as e:
        print(f"❌ Feil under lasting/lesing fra Cassandra: {e}")
        df_cassandra_read = None
else:
    print("❌ Kan ikke fortsette til Steg 3: Ingen data eller Spark er ikke aktiv.")
    df_cassandra_read = None

✅ SparkSession re-opprettet og konfigurert.


AttributeError: 'NoneType' object has no attribute 'empty'

In [ ]:
# --- Kjører Steg 4 (Spark-Transformasjon og Plotting) ---

if df_cassandra_read:
    print("\n--- Starter Spark-ekstraksjon og plotting ---")

    # 1. Spark: Ekstraher og transformer kolonner
    try:
        df_curated = df_cassandra_read.select(
                col("priceArea"), 
                col("productionGroup"), 
                col("startTime"), 
                col("quantityKwh")
            )
            
        # 2. Konverter 'startTime' til timestamp og legg til 'month'
        df_curated = df_curated.withColumn(
            "startTimeTS", 
            to_timestamp(col("startTime"), "yyyy-MM-dd'T'HH:mm:ssXXX")
        ).withColumn(
            "month", 
            df_curated["startTimeTS"].cast("date").substr(6, 2)
        )
        print("✅ Data kuratert for plotting.")

        # --- Plotting 1: Pai-diagram (NO1) ---
        CHOSEN_AREA = "NO1"
        
        # Aggreger data i Spark
        df_pie_spark = df_curated.filter(col("priceArea") == CHOSEN_AREA) \
            .groupBy("productionGroup") \
            .agg({"quantityKwh": "sum"}) \
            .withColumnRenamed("sum(quantityKwh)", "Total_KWh")

        df_pie_pd = df_pie_spark.toPandas()

        if not df_pie_pd.empty:
            plt.figure(figsize=(10, 8))
            plt.pie(
                df_pie_pd['Total_KWh'], 
                labels=df_pie_pd['productionGroup'], 
                autopct='%1.1f%%', 
                startangle=90,
                wedgeprops={'edgecolor': 'black'}
            )
            plt.title(f'Total Årsproduksjon ({CHOSEN_AREA}, {YEAR}) etter Produksjonsgruppe', fontsize=14)
            plt.tight_layout()
            plt.show()
            print(f"✅ Pai-diagram for {CHOSEN_AREA} generert.")
        else:
            print(f"❌ Ingen data for {CHOSEN_AREA} for å lage pai-diagram.")

        # --- Plotting 2: Linjeplott (Januar, NO1) ---
        
        df_line_spark = df_curated.filter(
            (col("priceArea") == CHOSEN_AREA) & (col("month") == "01")
        )

        df_line_pd = df_line_spark.select(
            "startTimeTS", 
            "productionGroup", 
            "quantityKwh"
        ).toPandas()

        if not df_line_pd.empty:
            plt.figure(figsize=(14, 7))
            sns.lineplot(
                data=df_line_pd, 
                x='startTimeTS', 
                y='quantityKwh', 
                hue='productionGroup'
            )
            plt.title(f'Produksjon etter Produksjonsgruppe - Januar {YEAR} ({CHOSEN_AREA})', fontsize=14)
            plt.xlabel("Tidspunkt")
            plt.ylabel("Kvantitet (kWh)")
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.legend(title='Produksjonsgruppe')
            plt.tight_layout()
            plt.show()
            print(f"✅ Linjeplott for Januar {CHOSEN_AREA} generert.")
        else:
            print(f"❌ Ingen data for Januar {CHOSEN_AREA} for å lage linjeplott.")

    except Exception as e:
        print(f"❌ Kritisk feil under Spark-ekstraksjon eller plotting: {e}")
        df_curated = None
else:
    print("❌ Kan ikke fortsette til Steg 4: Ingen data lest fra Cassandra.")
    df_curated = None

❌ Kan ikke fortsette til Steg 4: Ingen data lest fra Cassandra.


In [ ]:
# --- Kjører Steg 5 (Last til MongoDB) ---

# Gjenbruker df_curated fra Steg 4 (eller df_cassandra_read hvis 4 feilet)

if 'df_curated' in locals() and df_curated is not None:
    print("\n--- Starter MongoDB-lasting ---")
    
    # 1. Konverter Spark DataFrame til en liste av JSON-dokumenter 
    df_mongo_pd = df_curated.select(
        "priceArea", 
        "productionGroup", 
        "startTime", 
        "quantityKwh"
    ).toPandas() # Merk: Kun de fire kjerne-kolonnene

    records = df_mongo_pd.to_dict('records')

    # 2. Etabler MongoDB-tilkobling og last inn data
    try:
        client = MongoClient(MONGO_CONNECTION_STRING)
        db = client[MONGO_DB_NAME]
        collection = db[MONGO_COLLECTION_NAME]
        
        # Sletter eventuelle gamle data
        collection.delete_many({})
        
        if records:
            result = collection.insert_many(records)
            print(f"✅ MongoDB: Vellykket innsatt {len(result.inserted_ids)} dokumenter i kolleksjonen '{MONGO_COLLECTION_NAME}'.")
        else:
            print("❌ Ingen dokumenter å sette inn i MongoDB.")

        doc_count = collection.count_documents({})
        print(f"✅ MongoDB Verifisering: Totalt antall dokumenter i kolleksjonen: {doc_count}")

        client.close()

    except Exception as e:
        print(f"❌ Kritisk feil under tilkobling/lasting til MongoDB: {e}")
        print("Sjekk connection string, nettverkstilgang (IP whitelist) og brukernavn/passord.")

# Stopp Spark-sesjonen til slutt
if spark:
    spark.stop()
    print("✅ SparkSession stoppet.")